In [ ]:
!pip install -q pypdf langchain-text-splitters
print("✅ Kurulum tamam.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 29.4 MB/s eta 0:00:00
✅ Kurulum tamam.


In [ ]:
from pypdf import PdfReader
import re, json


PDF_PATH         = "/content/pathways.pdf"
YENI_CHUNKS_PATH = "/content/drive/MyDrive/endodonti_rag/outputs/chunks_temiz.jsonl"

reader = PdfReader(PDF_PATH)
print(f"Toplam sayfa: {len(reader.pages)}")

def clean_text(text):
    if not text: return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\u00ad", "", text)
    return text.strip()

IGNORE_PATTERNS = [
    "table of contents", "copyright", "about the editors",
    "contributors", "dedication", "list of tables",
    "list of illustrations", "instructions for online access",
    "cover image", "title page", "printed in canada",
    "content strategist", "publishing services manager"
]

def is_noisy_page(text):
    return any(p in text.lower() for p in IGNORE_PATTERNS)

def is_noisy_chunk(text):
    t = text.lower()
    if t.count("fig.") >= 2: return True
    if "list of illustrations" in t or "list of tables" in t: return True
    if t.count("references") >= 3 and len(t.split()) < 120: return True
    if re.search(r'\.\s+\d{1,3}\.\s+[A-Z]', text): return True
    if any(x in text for x in ["J Endod", "J Am Dent", "Oral Surg",
                                 "Dent Mater", "Int Endod", "et al:"]): return True
    return False

def quality_chunk(text):
    words = text.strip().split()
    if len(words) < 80: return False
    if len(set(w.lower() for w in words)) / max(len(words), 1) < 0.35: return False
    if text.strip().startswith("."): return False
    return True

from langchain_text_splitters import RecursiveCharacterTextSplitter

filtered_pages = []
for i, p in enumerate(reader.pages):
    if i % 100 == 0: print(f"  {i}/{len(reader.pages)} sayfa...")
    raw = p.extract_text()
    if not raw: continue
    txt = clean_text(raw)
    if not txt or is_noisy_page(txt): continue
    if (i + 1) < 29: continue
    filtered_pages.append({"page": i + 1, "text": txt})

print(f"Filtrelenmiş sayfa: {len(filtered_pages)}")


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []
for p in filtered_pages:
    for ch in splitter.split_text(p["text"]):
        ch = clean_text(ch)
        if not ch or is_noisy_chunk(ch) or not quality_chunk(ch): continue
        all_chunks.append({"page": p["page"], "text": ch})

print(f"Yeni chunk sayısı: {len(all_chunks)}")
print(f"\nÖrnek chunk:")
print(all_chunks[0]["text"][:300])


with open(YENI_CHUNKS_PATH, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"\n✅ Kaydedildi: {YENI_CHUNKS_PATH}")

Toplam sayfa: 3488
  0/3488 sayfa...
  100/3488 sayfa...
  200/3488 sayfa...
  300/3488 sayfa...
  400/3488 sayfa...
  500/3488 sayfa...
  600/3488 sayfa...
  700/3488 sayfa...
  800/3488 sayfa...
  900/3488 sayfa...
  1000/3488 sayfa...
  1100/3488 sayfa...
  1200/3488 sayfa...
  1300/3488 sayfa...
  1400/3488 sayfa...
  1500/3488 sayfa...
  1600/3488 sayfa...
  1700/3488 sayfa...
  1800/3488 sayfa...
  1900/3488 sayfa...
  2000/3488 sayfa...
  2100/3488 sayfa...
  2200/3488 sayfa...
  2300/3488 sayfa...
  2400/3488 sayfa...
  2500/3488 sayfa...
  2600/3488 sayfa...
  2700/3488 sayfa...
  2800/3488 sayfa...
  2900/3488 sayfa...
  3000/3488 sayfa...
  3100/3488 sayfa...
  3200/3488 sayfa...
  3300/3488 sayfa...
  3400/3488 sayfa...
Filtrelenmiş sayfa: 3419
Yeni chunk sayısı: 2046

Örnek chunk:
Introduction Louis H. Berman, Kenneth M. Hargreaves The foundation of the specialty of endodontics is a gift from the generations of great endodontists and researchers before us. They guided us w

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['finalverigizleme.docx', 'Notability', 'Veritabanı 3.Hafta', 'VERİ SUNUMU S.pptm.pptx', 'Hafta 10 - 26 Mayıs.zip', 'Hafta 10 - 26 Mayıs.zip (Unzipped Files)', '2 Haziran Sunumlar', 'Felaket Kurtarma ve Detaylı Veritabanı Yönetim İşlemleri.zip', 'class4.gslides', 'class5.gslides', 'class6.gslides', 'class7-8.gslides', 'class11-12.gslides', 'class13.gslides', 'pathways.pdf', 'esnek-dinozor-anahtarlik-2019-10-16.zip', 'öğrenci-öğretmen otomasyonu (2).zip', 'öğrenci-öğretmen otomasyonu (1).zip', 'öğrenci-öğretmen otomasyonu.zip', 'kacanRobot.zip', 'Classroom', 'Proje Bİlgi Formu.docx', '3- Sanal Sunucu Sistemleri (1).gdoc', '4-  Protokol, Topoloji, IP Adresleri, HTTP, HTTPS (1).pdf.zip', '4-  Protokol, Topoloji, IP Adresleri, HTTP, HTTPS.pdf.zip', 'Colab Notebooks', 'archive (1).zip', '3- Sanal Sunucu Sistemleri.gdoc', '4-  Protokol, Topoloji, IP Adresleri, HTTP, HTTPS.gdoc', '5- Firewall Sistemleri, VPN, Proxy.gdoc', '6-  LAN,WAN,WAF.gdoc', 'chunks (3).jsonl', 'Sistem Pr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
